In [ ]:
import os
import sys
import torch

REPO_URL = "https://github.com/KapachYonatan/Transformer-from-scratch.git"
REPO_DIR = "/content/Transformer-from-scratch"
CODE_DIR = f"{REPO_DIR}/code"

if not os.path.exists(REPO_DIR):
    os.system(f"git clone {REPO_URL} {REPO_DIR}")

# Install requirements (safe to run repeatedly)
os.system(f"pip install -q -r {REPO_DIR}/requirements.txt")

# Always set working directory and import path
os.chdir(CODE_DIR)
if CODE_DIR not in sys.path:
    sys.path.insert(0, CODE_DIR)

print("cwd:", os.getcwd())
print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

import data
import main
print("Imports OK")

cwd: /content/Transformer-from-scratch/code
CUDA available: True
GPU: Tesla T4
Imports OK


In [ ]:
from google.colab import drive
import os

drive.mount('/content/drive')
BASE_SAVE_PATH = '/content/drive/MyDrive/TransformerExperiments/'

Mounted at /content/drive


In [ ]:
import json
import torch
from pathlib import Path

In [ ]:
DATA_PATH = "../data/en/"
tokenized_data_path = Path("tokenized_data.pth")
tokenizer_path = Path("tokenizer.json")

if tokenized_data_path.exists() and tokenizer_path.exists():
  tokenized_data = torch.load(tokenized_data_path)
  tokenizer = data.CharTokenizer.load(str(tokenizer_path))
  print("Loaded cached tokenizer/tokenized data")
else:
  tokenizer, tokenized_data = data.load_data(DATA_PATH)
  torch.save(tokenized_data, tokenized_data_path)
  tokenizer.save(str(tokenizer_path))
  print("Created and cached tokenizer/tokenized data")
print("Docs:", len(tokenized_data), "Vocab:", tokenizer.vocab_size())

Starting data loading from ../data/en/...
Tokenizing file: ../data/en/input.txt...
Total documents loaded: 1. Total characters: 1115394.
Created and cached tokenizer/tokenized data
Docs: 1 Vocab: 66


In [ ]:
base_config = {
"seq_len": 128,
"batch_size": 64,
"n_layers": 6,
"n_heads": 8,
"embed_size": 192,
"learning_rate": 3e-4,
"gradient_clipping": 1.0,
"num_batches_to_train": 3500,
"val_interval": 100,
"sample_interval": 1000,
"with_residuals": True,
"use_pre_norm": True,
"init_scheme": "normal_0p02",
"weight_decay": 0.1,
"embedding_dropout": 0.15,
"attention_dropout": 0.15,
"self_attention_dropout": 0.15,
"scheduler_type": "none",
"adam_betas": [0.9, 0.95],
"early_stop_patience": 3,
"early_stop_delta": 0.0001
}

In [ ]:
phase_num = 5
base_model = {
    "exp_name": "reg_simpler_arch6"
}

variants = [
    base_model
]

experiments = main.build_experiments(base_config, variants)
print("Total experiments:", len(experiments))

Total experiments: 1


In [ ]:
from pathlib import Path
import json
import torch

all_results = []
save_root = f"{BASE_SAVE_PATH}/phase{phase_num}"

for cfg in experiments:
  print("\n=== Running:", cfg["exp_name"], "===")
  best_val = main.run_experiment(
  cfg,
  tokenizer,
  tokenized_data,
  base_save_path=save_root
  )
  all_results.append({
  "exp_name": cfg["exp_name"],
  "best_val_loss": float(best_val),
  "summary_path": str(Path(save_root) / cfg["exp_name"] / "summary.json")
  })
  if torch.cuda.is_available():
    torch.cuda.empty_cache()

all_results = sorted(all_results, key=lambda x: x["best_val_loss"])
leaderboard_path = Path(save_root) / "leaderboard_colab.json"
with open(leaderboard_path, "w") as f:
  json.dump(all_results, f, indent=2)

print("\nTop results:")
for r in all_results[:10]:
  print(r["exp_name"], "->", r["best_val_loss"])

print("Saved leaderboard:", leaderboard_path)


=== Running: reg_simpler_arch6 ===
Initializing experiment: reg_simpler_arch6 on cuda...
Train tokens: 1003854, Val tokens: 111540
Constructing Transformer model architecture...
Parameter count: 2.72M
Entering training loop for 3500 batches...
Model successfully moved to GPU. Starting training loop...
Batch [1] reached
Batch [2] reached
Batch [3] reached
Batch [4] reached
Batch [5] reached
Seen 10 batches. last loss is: 3.2637. lr: 0.0003. speed: 9.76 batches/sec
Seen 20 batches. last loss is: 2.9565. lr: 0.0003. speed: 9.84 batches/sec
Seen 30 batches. last loss is: 2.7333. lr: 0.0003. speed: 9.64 batches/sec
Seen 40 batches. last loss is: 2.6983. lr: 0.0003. speed: 9.62 batches/sec
Seen 50 batches. last loss is: 2.6394. lr: 0.0003. speed: 9.71 batches/sec
Seen 60 batches. last loss is: 2.5834. lr: 0.0003. speed: 9.43 batches/sec
Seen 70 batches. last loss is: 2.5416. lr: 0.0003. speed: 9.44 batches/sec
Seen 80 batches. last loss is: 2.5268. lr: 0.0003. speed: 9.39 batches/sec
Seen 9

## Training resume

In [ ]:
import copy

# Paths to previous run artifacts
old_exp_dir = Path(f"{BASE_SAVE_PATH}/phase6/best_model_low_lr")
saved_config_path = old_exp_dir / "config.json"
resume_ckpt = old_exp_dir / "best_model.pth"

# 1) Load old config as base
with open(saved_config_path, "r") as f:
    base_resume_config = json.load(f)

# 2) Update only what you want to change for resumed training
base_resume_config.update({
    "resume_from": str(resume_ckpt),
    "reset_optimizer_on_resume": True,   # True = apply new optimizer params
    "reset_scheduler_on_resume": False,   # True = start new scheduler behavior
    "learning_rate": 5e-8,
    "num_batches_to_train": 1500,
    "early_stop_patience": 4,
    "early_stop_delta": 1e-4,
})

# 3) Define short experiment variants
experiments = main.build_experiments(
    base_resume_config,
    [
        {"exp_name": "best_model_low_lr->4"}
    ],
)

# 4) Run exactly as before
phase_num = 6
all_results = []
save_root = f"{BASE_SAVE_PATH}/phase{phase_num}"

for cfg in experiments:
  print("\n=== Running:", cfg["exp_name"], "===")
  best_val = main.run_experiment(
  cfg,
  tokenizer,
  tokenized_data,
  base_save_path=save_root
  )
  all_results.append({
  "exp_name": cfg["exp_name"],
  "best_val_loss": float(best_val),
  "summary_path": str(Path(save_root) / cfg["exp_name"] / "summary.json")
  })
  if torch.cuda.is_available():
    torch.cuda.empty_cache()

all_results = sorted(all_results, key=lambda x: x["best_val_loss"])
leaderboard_path = Path(save_root) / "leaderboard_colab.json"
with open(leaderboard_path, "w") as f:
  json.dump(all_results, f, indent=2)

print("\nTop results:")
for r in all_results[:10]:
  print(r["exp_name"], "->", r["best_val_loss"])

print("Saved leaderboard:", leaderboard_path)


=== Running: best_model_low_lr->4 ===
Initializing experiment: best_model_low_lr->4 on cuda...
Train tokens: 1003854, Val tokens: 111540
Constructing Transformer model architecture...
Parameter count: 2.72M
Resuming from checkpoint: /content/drive/MyDrive/TransformerExperiments/phase6/best_model_low_lr/best_model.pth
Entering training loop for 1500 batches...
Model successfully moved to GPU. Starting training loop...
Batch [1] reached
Batch [2] reached
Batch [3] reached
Batch [4] reached
Batch [5] reached
Seen 10 batches. last loss is: 1.2778. lr: 5e-08. speed: 9.53 batches/sec
Seen 20 batches. last loss is: 1.2930. lr: 5e-08. speed: 9.57 batches/sec
Seen 30 batches. last loss is: 1.2637. lr: 5e-08. speed: 10.04 batches/sec
Seen 40 batches. last loss is: 1.2685. lr: 5e-08. speed: 9.91 batches/sec
Seen 50 batches. last loss is: 1.2867. lr: 5e-08. speed: 9.82 batches/sec
Seen 60 batches. last loss is: 1.2892. lr: 5e-08. speed: 9.73 batches/sec
Seen 70 batches. last loss is: 1.2252. lr: 

## Hebrew training

In [ ]:
DATA_PATH = "../data/he/"
tokenized_data_he_path = Path("tokenized_data_he.pth")
tokenizer_he_path = Path("tokenizer_he.json")

if tokenized_data_he_path.exists() and tokenizer_he_path.exists():
  tokenized_data_he = torch.load(tokenized_data_he_path)
  tokenizer_he = data.CharTokenizer.load(str(tokenizer_he_path))
  print("Loaded cached tokenizer/tokenized data")
else:
  tokenizer_he, tokenized_data_he = data.load_data(DATA_PATH)
  torch.save(tokenized_data_he, tokenized_data_he_path)
  tokenizer_he.save(str(tokenizer_he_path))
  print("Created and cached tokenizer/tokenized data")
print("Docs:", len(tokenized_data_he), "Vocab:", tokenizer_he.vocab_size())

Starting data loading from ../data/he/...
Tokenizing file: ../data/he/stripped_utf8__rachel__toreador.txt.clean.sentences.txt...
Tokenizing file: ../data/he/stripped_utf8__rachel__Rac113.txt.clean.sentences.txt...
Tokenizing file: ../data/he/stripped_utf8__bialik__bia121.txt.clean.sentences.txt...
Tokenizing file: ../data/he/stripped_utf8__bialik__bia_chds61.txt.clean.sentences.txt...
Tokenizing file: ../data/he/stripped_utf8__rachel__higiah.txt.clean.sentences.txt...
Tokenizing file: ../data/he/stripped_utf8__bialik__bia_chds53.txt.clean.sentences.txt...
Tokenizing file: ../data/he/stripped_utf8__bialik__bia_chds07.txt.clean.sentences.txt...
Tokenizing file: ../data/he/stripped_utf8__bialik__ezvon06.txt.clean.sentences.txt...
Tokenizing file: ../data/he/stripped_utf8__rachel__Rac048.txt.clean.sentences.txt...
Tokenizing file: ../data/he/stripped_utf8__bialik__bia098.txt.clean.sentences.txt...
Tokenizing file: ../data/he/stripped_utf8__rachel__Rac050.txt.clean.sentences.txt...
Tokenizi

#### Compute loss on best english model

In [ ]:
import os
import sys
import json
import math
import torch

# ---------- Paths ----------
REPO_ROOT = "/content/code-and-data"  # folder that contains code/
EN_EXP_DIR = Path(f"{BASE_SAVE_PATH}/phase6/best_model_low_lr")  # has best_model.pth, config.json, tokenizer.json

# Assumed to already exist in your notebook:
# tokenizer_he
# tokenized_data_he

# ---------- Imports ----------
sys.path.append(os.path.join(REPO_ROOT, "code"))
import data
import lm
from transformer import TransformerLM

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# ---------- Load English artifacts ----------
tok_en = data.CharTokenizer.load(os.path.join(EN_EXP_DIR, "tokenizer.json"))
with open(os.path.join(EN_EXP_DIR, "config.json"), "r") as f:
    cfg = json.load(f)

ckpt = torch.load(os.path.join(EN_EXP_DIR, "best_model.pth"), map_location=device)
if isinstance(ckpt, dict) and "model_state" in ckpt:
    ckpt = ckpt["model_state"]

# ---------- Build English model and load checkpoint ----------
embed_size = cfg["embed_size"]
model_en = TransformerLM(
    n_layers=cfg["n_layers"],
    n_heads=cfg["n_heads"],
    embed_size=embed_size,
    max_context_len=cfg["seq_len"],
    vocab_size=tok_en.vocab_size(),
    mlp_hidden_size=cfg.get("mlp_hidden_size", embed_size * 4),
    with_residuals=cfg.get("with_residuals", True),
    use_pre_norm=cfg.get("use_pre_norm", True),
    init_scheme=cfg.get("init_scheme", "xavier_uniform"),
    embedding_dropout=cfg.get("embedding_dropout", 0.0),
    attention_dropout=cfg.get("attention_dropout", 0.0),
    self_attention_dropout=cfg.get("self_attention_dropout", 0.0),
).to(device)
model_en.load_state_dict(ckpt)
model_en.eval()

# ---------- Build Hebrew-vocab model with same architecture ----------
model_he = TransformerLM(
    n_layers=cfg["n_layers"],
    n_heads=cfg["n_heads"],
    embed_size=embed_size,
    max_context_len=cfg["seq_len"],
    vocab_size=tokenizer_he.vocab_size(),
    mlp_hidden_size=cfg.get("mlp_hidden_size", embed_size * 4),
    with_residuals=cfg.get("with_residuals", True),
    use_pre_norm=cfg.get("use_pre_norm", True),
    init_scheme=cfg.get("init_scheme", "xavier_uniform"),
    embedding_dropout=cfg.get("embedding_dropout", 0.0),
    attention_dropout=cfg.get("attention_dropout", 0.0),
    self_attention_dropout=cfg.get("self_attention_dropout", 0.0),
).to(device)

# Copy all compatible weights except token embedding + output head (shape depends on vocab size)
state_en = model_en.state_dict()
skip = {
    "embed.token_embeddings.weight",
    "word_prediction.weight",
    "word_prediction.bias",
}
compatible = {k: v for k, v in state_en.items() if k not in skip}
model_he.load_state_dict(compatible, strict=False)

# ---------- Copy shared-character rows from EN vocab to HE vocab ----------
with torch.no_grad():
    shared = set(tok_en.stoi.keys()).intersection(set(tokenizer_he.stoi.keys()))
    for ch in shared:
        en_id = tok_en.stoi[ch]
        he_id = tokenizer_he.stoi[ch]
        model_he.embed.token_embeddings.weight[he_id].copy_(
            model_en.embed.token_embeddings.weight[en_id]
        )
        model_he.word_prediction.weight[he_id].copy_(
            model_en.word_prediction.weight[en_id]
        )
        model_he.word_prediction.bias[he_id].copy_(
            model_en.word_prediction.bias[en_id]
        )

print(f"Shared vocab symbols copied: {len(shared)} / {tokenizer_he.vocab_size()}")

# ---------- Loss on Hebrew tokenized data ----------
def compute_avg_loss(model, tokenizer, tokenized_data, batch_size=64):
    model.eval()
    pad_id = tokenizer.pad_id()
    seq_plus_one = model.max_context_len + 1

    def iter_windows():
        for seq in tokenized_data:
            if len(seq) == 0:
                continue
            for i in range(0, len(seq), seq_plus_one):
                chunk = seq[i:i + seq_plus_one]
                if len(chunk) < seq_plus_one:
                    chunk = chunk + [pad_id] * (seq_plus_one - len(chunk))
                yield chunk

    total_loss = 0.0
    total_tokens = 0

    with torch.no_grad():
        for batch in data.batch_items(iter_windows(), batch_size=batch_size):
            batch = batch.to(device)
            x, y = lm.batch_to_labeled_samples(batch)
            logits = model(x)
            loss = lm.compute_loss(logits, y, pad_id=pad_id)
            n_valid = int((y != pad_id).sum().item())
            total_loss += loss.item() * n_valid
            total_tokens += n_valid

    if total_tokens == 0:
        raise ValueError("No valid tokens for evaluation.")
    return total_loss / total_tokens

avg_loss = compute_avg_loss(model_he, tokenizer_he, tokenized_data_he, batch_size=64)
print(f"Hebrew loss (no retrain, transferred model): {avg_loss:.4f}")
print(f"Hebrew perplexity: {math.exp(avg_loss):.4f}")

Parameter count: 2.72M
Parameter count: 2.76M
Shared vocab symbols copied: 52 / 159
Hebrew loss (no retrain, transferred model): 6.3299
Hebrew perplexity: 561.0906


## Training Hebrew model

In [ ]:
base_config = {
"seq_len": 128,
"batch_size": 64,
"n_layers": 12,
"n_heads": 6,
"embed_size": 192,
"learning_rate": 1e-4,
"gradient_clipping": 1.0,
"num_batches_to_train": 5000,
"val_interval": 100,
"sample_interval": 200,
"with_residuals": True,
"use_pre_norm": True,
"init_scheme": "normal_0p02",
"weight_decay": 0,
"embedding_dropout": 0,
"attention_dropout": 0,
"self_attention_dropout": 0,
"scheduler_type": "none",
"adam_betas": [0.9, 0.95],
"early_stop_patience": 3,
"early_stop_delta": 0.0001
}

In [ ]:
phase_num = 1
base_model = {
    "exp_name": "deep_narrow2"
}

variants = [
    base_model,
    base_model | {"exp_name": "deep_narrow2_reg", "weight_decay": 0.1},
    base_model | {"exp_name": "deep_narrow2_reg_att", "attention_dropout": 0.1, "self_attention_dropout": 0.1}
]

experiments = main.build_experiments(base_config, variants)
print("Total experiments:", len(experiments))

Total experiments: 3


In [ ]:
from pathlib import Path
import json
import torch

all_results = []
save_root = f"{BASE_SAVE_PATH}/Hebrew/phase{phase_num}"

for cfg in experiments:
  print("\n=== Running:", cfg["exp_name"], "===")
  best_val = main.run_experiment(
  cfg,
  tokenizer_he,
  tokenized_data_he,
  base_save_path=save_root
  )
  all_results.append({
  "exp_name": cfg["exp_name"],
  "best_val_loss": float(best_val),
  "summary_path": str(Path(save_root) / cfg["exp_name"] / "summary.json")
  })
  if torch.cuda.is_available():
    torch.cuda.empty_cache()

all_results = sorted(all_results, key=lambda x: x["best_val_loss"])
leaderboard_path = Path(save_root) / "leaderboard_colab.json"
with open(leaderboard_path, "w") as f:
  json.dump(all_results, f, indent=2)

print("\nTop results:")
for r in all_results[:10]:
  print(r["exp_name"], "->", r["best_val_loss"])

print("Saved leaderboard:", leaderboard_path)


=== Running: deep_narrow2 ===
Initializing experiment: deep_narrow2 on cuda...
Train tokens: 725364, Val tokens: 62213
Constructing Transformer model architecture...
Parameter count: 5.42M
Entering training loop for 5000 batches...
Model successfully moved to GPU. Starting training loop...
Batch [1] reached
Batch [2] reached
Batch [3] reached
Batch [4] reached
Batch [5] reached
Seen 10 batches. last loss is: 4.1261. lr: 0.0001. speed: 4.99 batches/sec
Seen 20 batches. last loss is: 3.7758. lr: 0.0001. speed: 4.67 batches/sec
Seen 30 batches. last loss is: 3.5483. lr: 0.0001. speed: 4.56 batches/sec
Seen 40 batches. last loss is: 3.3583. lr: 0.0001. speed: 4.51 batches/sec
Seen 50 batches. last loss is: 3.2288. lr: 0.0001. speed: 4.38 batches/sec
Seen 60 batches. last loss is: 3.0991. lr: 0.0001. speed: 4.34 batches/sec
Seen 70 batches. last loss is: 2.9886. lr: 0.0001. speed: 4.31 batches/sec
Seen 80 batches. last loss is: 2.9456. lr: 0.0001. speed: 4.33 batches/sec
Seen 90 batches. l

## Hebrew training resume

In [ ]:
import copy

# Paths to previous run artifacts
old_exp_dir = Path(f"{BASE_SAVE_PATH}/Hebrew/phase2/best_model_low_lr->4e-6")
saved_config_path = old_exp_dir / "config.json"
resume_ckpt = old_exp_dir / "best_model.pth"

# 1) Load old config as base
with open(saved_config_path, "r") as f:
    base_resume_config = json.load(f)
phase_num = 2

# 2) Update only what you want to change for resumed training
base_resume_config.update({
    "resume_from": str(resume_ckpt),
    "reset_optimizer_on_resume": True,   # True = apply new optimizer params
    "reset_scheduler_on_resume": False,   # True = start new scheduler behavior
    "learning_rate": 5e-8,
    "num_batches_to_train": 1500,
    "early_stop_patience": 4,
    "early_stop_delta": 1e-4,
})

# 3) Define short experiment variants
experiments = main.build_experiments(
    base_resume_config,
    [
        {"exp_name": "best_model_low_lr->5e-8"}
    ],
)

# 4) Run exactly as before
all_results = []
save_root = f"{BASE_SAVE_PATH}/Hebrew/phase{phase_num}"

for cfg in experiments:
  print("\n=== Running:", cfg["exp_name"], "===")
  best_val = main.run_experiment(
  cfg,
  tokenizer_he,
  tokenized_data_he,
  base_save_path=save_root
  )
  all_results.append({
  "exp_name": cfg["exp_name"],
  "best_val_loss": float(best_val),
  "summary_path": str(Path(save_root) / cfg["exp_name"] / "summary.json")
  })
  if torch.cuda.is_available():
    torch.cuda.empty_cache()

all_results = sorted(all_results, key=lambda x: x["best_val_loss"])
leaderboard_path = Path(save_root) / "leaderboard_colab.json"
with open(leaderboard_path, "w") as f:
  json.dump(all_results, f, indent=2)

print("\nTop results:")
for r in all_results[:10]:
  print(r["exp_name"], "->", r["best_val_loss"])

print("Saved leaderboard:", leaderboard_path)


=== Running: best_model_low_lr->5e-8 ===
Initializing experiment: best_model_low_lr->5e-8 on cuda...
Train tokens: 725364, Val tokens: 62213
Constructing Transformer model architecture...
Parameter count: 5.42M
Resuming from checkpoint: /content/drive/MyDrive/TransformerExperiments/Hebrew/phase2/best_model_low_lr->4e-6/best_model.pth
Entering training loop for 1500 batches...
Model successfully moved to GPU. Starting training loop...
Batch [1] reached
Batch [2] reached
Batch [3] reached
Batch [4] reached
Batch [5] reached
Seen 10 batches. last loss is: 1.7549. lr: 5e-08. speed: 4.72 batches/sec
Seen 20 batches. last loss is: 1.7581. lr: 5e-08. speed: 4.45 batches/sec
Seen 30 batches. last loss is: 1.6802. lr: 5e-08. speed: 4.38 batches/sec
Seen 40 batches. last loss is: 1.8178. lr: 5e-08. speed: 4.30 batches/sec
Seen 50 batches. last loss is: 1.7652. lr: 5e-08. speed: 4.23 batches/sec
Seen 60 batches. last loss is: 1.7204. lr: 5e-08. speed: 4.18 batches/sec
Seen 70 batches. last loss 